In [1]:
%cd ~/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
%load_ext autoreload
%autoreload 2
import sys
print(sys.executable)

/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/.venv/bin/python


In [2]:
%%writefile pages/home.py
import streamlit as st

from src import config, ui
from src.data import load, load_half

ui.setup("好球帶研究室")
teams, batters, pitchers, const, lg = load()

st.markdown(f"""
<div class="card" style="padding:28px">
  <p class="eyebrow">2026 中華職棒 · {config.TS_TEAMS[0]} vs {config.TS_TEAMS[1]}</p>
  <h2 style="margin:6px 0 10px">用進階數據，看懂這個十月。</h2>
  <p style="color:#8C9BB0;max-width:36em">從 wOBA 到 FIP，從被低估的球員到台灣大賽勝率，
  這裡的每個結論都有數據根據。</p>
</div>""", unsafe_allow_html=True)

c1, c2, c3 = st.columns(3)
c1.page_link("pages/academy.py", label="數據學院：每個數據的白話解釋與計算機")
c2.page_link("pages/watch.py", label="球季觀察室：誰被低估？誰是泡沫？")
c3.page_link("pages/predict.py", label="台灣大賽預測：自選先發與打線算勝率")

Overwriting pages/home.py


In [3]:
%%writefile -a pages/home.py
st.header('球季數據亮點')
qb=batters[batters['PA']>=300]
qp=pitchers[pitchers['IP']>=50]
top_woba=qb.nlargest(1,'wOBA_calc').iloc[0]
gap=qp.assign(d=qp['ERA']-qp['FIP_calc']).nlargest(1,'d').iloc[0]
top_k = qp[qp["role"] == "先發"].nlargest(1, "Kp").iloc[0]
top_babip = qb.nlargest(1, "BABIP").iloc[0]
cols = st.columns(4)
with cols[0]:
    ui.kpi("wOBA 最高", ui.f3(top_woba["wOBA_calc"]), f"{top_woba['team']}・{top_woba['name']}")
with cols[1]:
    ui.kpi("ERA 與 FIP 落差最大", f"+{gap['d']:.2f}", f"{gap['team']}・{gap['name']}", ui.chip("運氣差", "cold"))
with cols[2]:
    ui.kpi("先發三振率最高", f"{top_k['Kp']:.1f}%", f"{top_k['team']}・{top_k['name']}", ui.chip("聯盟第 1", "good"))
with cols[3]:
    ui.kpi("BABIP 最高", ui.f3(top_babip["BABIP"]), f"{top_babip['team']}・{top_babip['name']}",
           ui.chip("可能有運氣成分", "hot"))

Appending to pages/home.py


In [4]:
%%writefile -a pages/home.py
st.header("全年戰績")
show = teams[["rank", "team", "W", "L", "T", "PCT", "RS", "RA", "pythag", "luck"]]
st.dataframe(show, hide_index=True, width="stretch", column_config={
    "rank": "排名", "team": "球隊", "PCT": st.column_config.NumberColumn("勝率", format="%.3f"),
    "RS": "得分", "RA": "失分",
    "pythag": st.column_config.NumberColumn("畢氏勝率", format="%.3f"),
    "luck": st.column_config.NumberColumn("實際 − 畢氏", format="%+.3f",
                                          help="正數代表贏得比得失分差應有的多，可能有運氣成分")})
st.caption("資料來源：野球革命 Rebas、中華職棒官方網站")

Appending to pages/home.py


In [5]:
%%writefile -a src/data.py
@st.cache_data
def load_half():
    """中職官網的上下半季戰績（Day 25）；沒有檔案就回傳 None。"""
    path = CLEAN / "half_standings.csv"
    return pd.read_csv(path) if path.exists() else None

Appending to src/data.py


In [6]:
!python scripts/update_data.py

半季戰績來源：中職官網；交叉比對全部一致
球隊 6、打者 183、投手 181；wOBA 倍數 = 1.001、FIP 常數 = 3.02
逐打席 27268 筆；聯盟 wOBA 0.319


In [7]:
!git status
!du -sh data/clean data/raw

On branch main
Your branch is ahead of 'origin/main' by 6 commits.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   D12.ipynb
	modified:   pages/home.py
	modified:   src/data.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	D13.ipynb

no changes added to commit (use "git add" and/or "git commit -a")
3.2M	data/clean
4.1M	data/raw


In [8]:
!git add .
!git commit -m "Day 26: 部署前整理"
!git push

[main 058ad7b] Day 26: 部署前整理
 Committer: Leo Chen <leochen@LeodeMacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 4 files changed, 272 insertions(+), 6 deletions(-)
 create mode 100644 D13.ipynb
Enumerating objects: 82, done.
Counting objects: 100% (82/82), done.
Delta compression using up to 8 threads
Compressing objects: 100% (59/59), done.
Writing objects: 100% (62/62), 39.09 KiB | 9.77 MiB/s, done.
Total 62 (delta 39), reused 0 (delta 0), pack-reused 0 (from 0)
remote: Resolving deltas: 100% (39/39), completed with 13 local objects.
To https://github.com/leo8507241/baseballlab.git
   

In [9]:
%%writefile README.md
---
title: 好球帶研究室
emoji: ⚾
colorFrom: yellow
colorTo: blue
sdk: streamlit
sdk_version: 1.51.0
python_version: "3.12"
app_file: app.py
pinned: false
---

Writing README.md


In [10]:
%%writefile scripts/deploy.py
"""把網站上傳到 Hugging Face Space。用法：python scripts/deploy.py"""
from pathlib import Path

from huggingface_hub import HfApi

ROOT = Path(__file__).resolve().parent.parent
SPACE = "<你的帳號>/haoqiudai-lab"

HfApi().upload_folder(
    folder_path=str(ROOT), repo_id=SPACE, repo_type="space",
    ignore_patterns=[".venv/*", "data/raw/*", "tutorial/*", "tests/*", ".git/*", "notebooks/*",
                     "**/__pycache__/*", "**/.ipynb_checkpoints/*", "explore*.py", ".DS_Store"],
    commit_message="更新網站",
)
print(f"上傳完成：https://huggingface.co/spaces/{SPACE}")

Writing scripts/deploy.py


In [11]:
!python scripts/deploy.py

Traceback (most recent call last):
  File "/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/scripts/deploy.py", line 4, in <module>
    from huggingface_hub import HfApi
ModuleNotFoundError: No module named 'huggingface_hub'


In [13]:
!git add README.md scripts/deploy.py
!git commit -m "新增 README 和部署腳本"
!git push

[main 5018933] 新增 README 和部署腳本
 Committer: Leo Chen <leochen@LeodeMacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 2 files changed, 30 insertions(+)
 create mode 100644 README.md
 create mode 100644 scripts/deploy.py
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 8 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.09 KiB | 1.09 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/leo8507241/baseballl

In [14]:
!python scripts/update_data.py
!git add .
!git commit -m "Day 27: 上線除錯"
!git push

第1次失敗 https://www.rebas.tw/api/seasons/CPBL-2026-oB/games 原因：HTTPSConnectionPool(host='www.rebas.tw', port=443): Max retries exceeded with url: /api/seasons/CPBL-2026-oB/games (Caused by NameResolutionError("HTTPSConnection(host='www.rebas.tw', port=443): Failed to resolve 'www.rebas.tw' ([Errno 8] nodename nor servname provided, or not known)"))
第2次失敗 https://www.rebas.tw/api/seasons/CPBL-2026-oB/games 原因：HTTPSConnectionPool(host='www.rebas.tw', port=443): Max retries exceeded with url: /api/seasons/CPBL-2026-oB/games (Caused by NameResolutionError("HTTPSConnection(host='www.rebas.tw', port=443): Failed to resolve 'www.rebas.tw' ([Errno 8] nodename nor servname provided, or not known)"))
第3次失敗 https://www.rebas.tw/api/seasons/CPBL-2026-oB/games 原因：HTTPSConnectionPool(host='www.rebas.tw', port=443): Max retries exceeded with url: /api/seasons/CPBL-2026-oB/games (Caused by NameResolutionError("HTTPSConnection(host='www.rebas.tw', port=443): Failed to resolve 'www.rebas.tw' ([Errno 8] no